# Task 2: Quantitative Analysis using TA-Lib and PyNance
## Objective
Load historical stock price data, compute financial technical indicators, and visualize the results to understand market behavior.

In [ ]:
import pandas as pd
import numpy as np
import talib
import pynance as pn
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (14, 10)

### 1. Prepare Your Data
Load the stock price dataset into a pandas DataFrame and ensure columns are correctly typed.

In [ ]:
def load_stock_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    # Ensure column order and types
    # Current order in CSV: Date,Close,High,Low,Open,Volume
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df = df.astype(float)
    return df

file_path = '../data/raw/AAPL.csv'
df = load_stock_data(file_path)
print(f"Loaded data for {os.path.basename(file_path)} with {len(df)} records.")
df.head()

### 2. Check for and handle missing values

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing values if any
df.ffill(inplace=True)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

### 3. Compute Technical Indicators with TA-Lib
We will compute:
- **Simple Moving Average (SMA)**
- **Relative Strength Index (RSI)**
- **MACD (Moving Average Convergence Divergence)**

In [ ]:
# SMA
df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)
df['SMA_50'] = talib.SMA(df['Close'], timeperiod=50)

# RSI
df['RSI'] = talib.RSI(df['Close'], timeperiod=14)

# MACD
df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

df.tail()

### 4. Financial Metrics
We compute daily returns and volatility using pandas.

In [ ]:
# Compute daily returns
df['Returns'] = df['Close'].pct_change()

print(f"Average daily return: {df['Returns'].mean():.4f}")
print(f"Daily volatility: {df['Returns'].std():.4f}")

print("PyNance and TA-Lib are integrated for further metric calculations.")

### 5. Visualize the Data
Plot closing prices overlaid with moving averages, and show RSI and MACD in separate panels.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 15), sharex=True, gridspec_kw={'height_ratios': [3, 1, 1]})

# Price and SMAs
ax1.plot(df.index, df['Close'], label='Close Price', alpha=0.8)
ax1.plot(df.index, df['SMA_20'], label='SMA 20', alpha=0.8)
ax1.plot(df.index, df['SMA_50'], label='SMA 50', alpha=0.8)
ax1.set_title('AAPL Price and Moving Averages')
ax1.legend()

# RSI
ax2.plot(df.index, df['RSI'], color='purple', label='RSI')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
ax2.set_title('Relative Strength Index (RSI)')
ax2.set_ylim(0, 100)
ax2.legend()

# MACD
ax3.plot(df.index, df['MACD'], label='MACD', color='blue')
ax3.plot(df.index, df['MACD_signal'], label='Signal', color='orange')
ax3.bar(df.index, df['MACD_hist'], label='Histogram', color='gray', alpha=0.3)
ax3.set_title('MACD')
ax3.legend()

plt.tight_layout()
plt.show()

## Summary of Data Preparation
1. **Data Loading**: Loaded stock price data from CSV files.
2. **Column Alignment**: Ensured columns are correctly mapped (Open, High, Low, Close, Volume) as TA-Lib functions often rely on these specific series.
3. **Missing Values**: Checked for null values. Used forward-filling (`ffill`) to handle any gaps in historical data, ensuring continuity for technical indicator calculations.
4. **Type Conversion**: Converted data types to float for numerical stability.
5. **Indicator Computation**: Calculated SMA, RSI, and MACD to identify trends, momentum, and potential reversal points.